# 春第 4 回｜グループ集計と割合

予定どおりの行先に 15 分未満の遅れで着いた便を数えます。空港や路線ごとに集計し，どの便で割った割合かを読みます。

最初に本文の例を一緒に実行し，後半では自分で条件を選んで約 20 分の実習に取り組みます。

## 準備

Colab では「ドライブにコピーを保存」で自分用のノートを作り，上から準備セルを実行します。CSV は公開 URL から読み込むため，アップロードは不要です。ローカルでは，実習 ZIP を展開してノートブックと CSV を同じフォルダーに置くと，手元のファイルを使えます。

最初のセルはファイルの置き場所に対応する準備です。そのまま実行してください。出力は `32347` です。

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

file_name = "nyc-flights-2024-01.csv"
csv_path = Path("data") / file_name
if not csv_path.is_file():
    csv_path = Path("../data") / file_name
if not csv_path.is_file():
    csv_path = Path(file_name)
if not csv_path.is_file():
    csv_path = "https://raw.githubusercontent.com/sohei-takagi/python-ds-course-materials/main/data/" + file_name

flights = pd.read_csv(csv_path)
len(flights)

## 本文と一緒に実行

問いを読んでから，コードを上から順に実行します。「提供した作図セル」はそのまま実行し，軸や点，棒の位置を読みます。

### 100% を作った便を調べる

ロサンゼルス（LAX）行きの定時到着率を見ます。ラガーディア（LGA）の 100% が何便からできたか，図の次に元の表で確認します。

In [ ]:
# 提供した作図セル。そのまま実行します。
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.ticker import PercentFormatter

opening_flights = flights.copy()
opening_flights["on_time"] = (
    (opening_flights["cancelled"] == 0)
    & (opening_flights["diverted"] == 0)
    & (opening_flights["arrival_delay"] < 15)
)
opening_lax = opening_flights.loc[opening_flights["destination"] == "LAX"]
opening_rates = opening_lax.groupby("origin")["on_time"].mean()
opening_rates = opening_rates.reindex(["LGA", "JFK", "EWR"])

opening_figure, opening_axis = plt.subplots(layout="constrained")
opening_bars = opening_axis.barh(
    opening_rates.index, opening_rates, color=["#b15334", "#286b70", "#286b70"]
)
opening_axis.bar_label(
    opening_bars, labels=[f"{rate:.1%}" for rate in opening_rates], padding=6
)
opening_axis.set(xlim=(0, 1.12), xticks=[0, .25, .5, .75, 1],
                 xlabel="On-time arrival rate")
opening_axis.xaxis.set_major_formatter(PercentFormatter(xmax=1))
opening_axis.invert_yaxis()
opening_axis.spines[["top", "right"]].set_visible(False)
plt.show()

In [ ]:
import pandas as pd

lecture_flights = flights.copy()
lax_preview = lecture_flights.loc[lecture_flights["destination"] == "LAX"]
lga_lax = lax_preview.loc[lax_preview["origin"] == "LGA"]
lga_lax[["date", "origin", "destination", "scheduled_departure", "arrival_delay"]]

### 一便ずつ判定し，空港ごとに数える

欠航と到着先変更がなく，遅れが 15 分未満の便を定時到着とします。6 便の表から，各条件と `on_time` を読みます。

In [ ]:
not_cancelled = lecture_flights["cancelled"] == 0
not_diverted = lecture_flights["diverted"] == 0
within_limit = lecture_flights["arrival_delay"] < 15
lecture_flights["on_time"] = not_cancelled & not_diverted & within_limit

In [ ]:
sample_labels = [128, 408, 1011, 4671, 5247, 9117]
sample_columns = ["origin", "arrival_delay", "cancelled", "diverted", "on_time"]
sample = lecture_flights.loc[sample_labels, sample_columns]
sample

`size()` が数えた行数と，`sum()` が数えた定時到着便を比べます。同じ手順で，LAX 行き全体の分子と分母を作ります。

In [ ]:
sample_groups = sample.groupby("origin")["on_time"]
sample_groups.size()

In [ ]:
sample_groups.sum()

In [ ]:
lax = lecture_flights.loc[lecture_flights["destination"] == "LAX"]
by_origin_lax = lax.groupby("origin")["on_time"]
lax_scheduled = by_origin_lax.size()
lax_on_time = by_origin_lax.sum()

In [ ]:
lax_counts = lax_scheduled.to_frame("scheduled")
lax_counts["on_time"] = lax_on_time
lax_counts["rate"] = lax_counts["on_time"] / lax_counts["scheduled"]
lax_counts.round({"rate": 4})

### 同じ 100% でも便数を読む

ラガーディア発を行先別にまとめます。散布図では右にいくほど予定便が多く，上にいくほど定時到着率が高くなります。100% に当たる行を次の表で確かめます。

In [ ]:
lga = lecture_flights.loc[lecture_flights["origin"] == "LGA"]
by_destination = lga.groupby("destination")["on_time"]
lga_routes = by_destination.size().to_frame("scheduled")
lga_routes["on_time"] = by_destination.sum()
lga_routes["rate"] = lga_routes["on_time"] / lga_routes["scheduled"]

In [ ]:
# 提供した作図セル。そのまま実行します。
import matplotlib.pyplot as plt

lga_routes.plot.scatter(
    x="scheduled", y="rate", ylim=(0, 1.05),
    xlabel="Scheduled flights", ylabel="On-time arrival rate",
    color="#286b70",
)
plt.show()

In [ ]:
lga_routes.loc[lga_routes["rate"] == 1, ["on_time", "scheduled"]]

### 分子をそろえて分母を替える

予定便の件数と，到着の遅れが記録された便の件数を各空港で求めます。定時到着便数を二通りの分母で割り，上位の空港がどう変わるかを図で確認します。

In [ ]:
groups = lecture_flights.groupby("origin")
airport_counts = groups.size().to_frame("scheduled")
airport_counts["on_time"] = groups["on_time"].sum()
airport_counts["recorded"] = groups["arrival_delay"].count()
airport_counts

In [ ]:
airport_counts["rate_scheduled"] = airport_counts["on_time"] / airport_counts["scheduled"]
airport_counts["rate_recorded"] = airport_counts["on_time"] / airport_counts["recorded"]
airport_counts.round({"rate_scheduled": 4, "rate_recorded": 4})

In [ ]:
# 提供した作図セル。そのまま実行します。
from matplotlib.ticker import PercentFormatter

# 直前までに作った airport_counts の二つの率を描く。
comparison_figure, comparison_axis = plt.subplots(layout="constrained")
airport_labels = airport_counts.index
scheduled_rates = airport_counts["rate_scheduled"]
recorded_rates = airport_counts["rate_recorded"]
comparison_axis.hlines(airport_labels, scheduled_rates, recorded_rates,
                       color="#999999", linewidth=1.4)
comparison_axis.scatter(scheduled_rates, airport_labels, color="#286b70", s=60,
                        zorder=3, label="All scheduled flights")
comparison_axis.scatter(recorded_rates, airport_labels, color="#b15334", marker="s",
                        s=60, zorder=3, label="Flights with arrival delay recorded")
comparison_axis.set(xlim=(.72, .82), xticks=[.72, .74, .76, .78, .80, .82],
                    xlabel="On-time arrival rate")
comparison_axis.xaxis.set_major_formatter(PercentFormatter(xmax=1, decimals=0))
comparison_axis.margins(y=.20)
comparison_axis.invert_yaxis()
comparison_axis.legend(loc="upper center", bbox_to_anchor=(.5, -.23),
                       ncol=2, frameon=False)
comparison_axis.spines[["top", "right"]].set_visible(False)
plt.show()

到着の遅れが空欄の便を取り出し，欠航・到着先変更がどの空港に何件あったかを見ます。

In [ ]:
missing_arrivals = lecture_flights.loc[lecture_flights["arrival_delay"].isna()]
missing_arrivals.groupby("origin")[["cancelled", "diverted"]].sum()

### 路線を合わせるときの便数の重み

EWR と JFK からオーランド（MCO）とサンフランシスコ（SFO）に向かう路線を比べます。路線別の二つの率と，分子と分母を足して作った率を読みます。

In [ ]:
pair_flights = lecture_flights.loc[
    (lecture_flights["origin"] == "EWR") | (lecture_flights["origin"] == "JFK")
]
mco = pair_flights.loc[pair_flights["destination"] == "MCO"]
sfo = pair_flights.loc[pair_flights["destination"] == "SFO"]

In [ ]:
mco_by_origin = mco.groupby("origin")["on_time"]
mco_counts = mco_by_origin.size().to_frame("scheduled")
mco_counts["on_time"] = mco_by_origin.sum()
mco_counts["rate"] = mco_counts["on_time"] / mco_counts["scheduled"]
mco_counts

In [ ]:
sfo_by_origin = sfo.groupby("origin")["on_time"]
sfo_counts = sfo_by_origin.size().to_frame("scheduled")
sfo_counts["on_time"] = sfo_by_origin.sum()
sfo_counts["rate"] = sfo_counts["on_time"] / sfo_counts["scheduled"]
sfo_counts

In [ ]:
combined_scheduled = mco_counts["scheduled"] + sfo_counts["scheduled"]
combined_on_time = mco_counts["on_time"] + sfo_counts["on_time"]
combined_rate = combined_on_time / combined_scheduled
combined_rate

図で，空港ごとに MCO 行きがどれくらい含まれるか読みます。実際の便数の割合を重みにした計算と，二路線に同じ重みを置いた計算を比べます。

In [ ]:
# 提供した作図セル。そのまま実行します。
from matplotlib.ticker import PercentFormatter

# ここまでの mco_counts，sfo_counts，combined_scheduled を使う。
mix_figure, mix_axis = plt.subplots(layout="constrained")
mix_origins = combined_scheduled.index
mco_shares = mco_counts["scheduled"] / combined_scheduled
sfo_shares = sfo_counts["scheduled"] / combined_scheduled
mco_bars = mix_axis.barh(mix_origins, mco_shares, color="#286b70", label="MCO")
sfo_bars = mix_axis.barh(
    mix_origins, sfo_shares, left=mco_shares, color="#825039", label="SFO"
)
mix_axis.bar_label(
    mco_bars, labels=[f"{share:.1%}" for share in mco_shares],
    label_type="center", color="white"
)
mix_axis.bar_label(
    sfo_bars, labels=[f"{share:.1%}" for share in sfo_shares],
    label_type="center", color="white"
)
mix_axis.set(xlim=(0, 1), xticks=[0, .25, .5, .75, 1],
             xlabel="Share of scheduled flights on the two routes")
mix_axis.xaxis.set_major_formatter(PercentFormatter(xmax=1, decimals=0))
mix_axis.invert_yaxis()
mix_axis.legend(loc="upper center", bbox_to_anchor=(.5, -.24), ncol=2, frameon=False)
mix_axis.spines[["top", "right"]].set_visible(False)
plt.show()

In [ ]:
mco_weight = mco_counts["scheduled"] / combined_scheduled
sfo_weight = 1 - mco_weight
weighted_rate = mco_weight * mco_counts["rate"] + sfo_weight * sfo_counts["rate"]
weighted_rate

In [ ]:
equal_mix_rate = 0.5 * mco_counts["rate"] + 0.5 * sfo_counts["rate"]
equal_mix_rate

In [ ]:
# 提供した作図セル。そのまま実行します。
from matplotlib.lines import Line2D

weight_figure, weight_axis = plt.subplots(layout="constrained")
weight_axis.axvline(.5, linestyle=":", linewidth=1.2, color="#777777")
for origin, color, line_style in (
    ("EWR", "#286b70", "-"),
    ("JFK", "#9c3d22", "--"),
):
    endpoint_rates = [sfo_counts.loc[origin, "rate"], mco_counts.loc[origin, "rate"]]
    weight_axis.plot([0, 1], endpoint_rates, color=color, linestyle=line_style,
                    linewidth=1.9, label=origin)
    weight_axis.scatter(mco_weight.loc[origin], combined_rate.loc[origin],
                        color=color, marker="o", s=55, zorder=3)
    weight_axis.scatter(.5, equal_mix_rate.loc[origin], color=color,
                        marker="s", s=55, zorder=3)
line_handles, _ = weight_axis.get_legend_handles_labels()
example_points = [
    Line2D([], [], color="#444444", marker="o", linestyle="none", label="Actual mix"),
    Line2D([], [], color="#444444", marker="s", linestyle="none", label="Equal mix"),
]
weight_axis.legend(handles=line_handles + example_points, ncol=2,
                   loc="upper right", frameon=False)
weight_axis.set(xlim=(-.02, 1.02), ylim=(.65, .80),
                xticks=[0, .25, .5, .75, 1], yticks=[.65, .70, .75, .80],
                xlabel="Weight given to MCO", ylabel="Combined on-time arrival rate")
weight_axis.xaxis.set_major_formatter(PercentFormatter(xmax=1, decimals=0))
weight_axis.yaxis.set_major_formatter(PercentFormatter(xmax=1, decimals=0))
weight_axis.spines[["top", "right"]].set_visible(False)
plt.show()

## Jupyter 実習（約 20 分）

同じ定時到着便数でも，何で割るかによって答える問いが変わります。今回は 2024 年 1 月の記録で ATL（アトランタ）か MCO（オーランド）を選び，依頼に合う割合を計算します。

共通課題は 1〜3 の約 20 分です。条件を増やして順位を逆転させることが目的ではありません。**何を分母に選び，その率が何を表すか**を説明してください。末尾の追加課題は任意です。

次の判定は本文と同じです。欠航せず，到着先も変更されず，到着の遅れが 15 分未満なら `True` にします。この判定を足したものが，今回の二つの依頼に共通する分子です。

In [ ]:
flights["on_time"] = (
    (flights["cancelled"] == 0)
    & (flights["diverted"] == 0)
    & (flights["arrival_delay"] < 15)
)
flights[["origin", "arrival_delay", "on_time"]].head()

### 1. 依頼と行先を選ぶ（5 分）

行先は ATL か MCO，依頼は A か B を一つずつ選びます。

| 依頼 | 知りたいこと |
|---|---|
| A | この行先へ出る予定だった便のうち，何割が定時に目的地へ着いたか。 |
| B | この行先への到着の遅れが記録された便に限ると，何割が定時だったか。 |

選んだ依頼（A／B）：

分母に含める便（自分の言葉で）：

次の `destination` に行先を指定して，集計の材料を表示します。`scheduled` は予定便数，`recorded` は到着の遅れが記録された便数，`on_time` は定時到着便数です。空港を一つ選び，この 3 列の値を読み比べてください。

In [ ]:
destination = "ATL"  # MCO を選んだ人はここを変更
route = flights.loc[flights["destination"] == destination]
groups = route.groupby("origin")

counts = groups.size().to_frame("scheduled")
counts["recorded"] = groups["arrival_delay"].count()
counts["on_time"] = groups["on_time"].sum()
counts

### 2. 分母を選んで計算する（9 分）

`groups` は，選んだ行先の便を出発空港ごとにまとめたものです。依頼に合う分母を，`denominator = ...` という一行で書きます。本文の `size()` と `count()` を参照できます。

書いたら，その下の 4 行も先頭の `#` を外して実行してください。`denominator` と `on_time` は，同じ空港名の行どうしで割り算されます。元の `counts` は残したまま，結果を `report` にまとめます。

`count()` を使う場合は，**どの列の値の有無を数えるか**にも注意してください。`on_time` は，元の到着の遅れが欠損でも `False` が入る列です。

In [ ]:
# ここに denominator = ... の一行を書く

# report = denominator.to_frame("denominator")
# report["on_time"] = counts["on_time"]
# report["rate"] = report["on_time"] / report["denominator"]
# report.round({"rate": 4})

<details>
<summary>分母の式のヒント</summary>

全予定便を数えるなら `groups.size()`，到着の遅れに値がある便を数えるなら `groups["arrival_delay"].count()` です。自分が選んだ依頼に合う方を `denominator` へ代入します。

</details>

結果の表ができたら，次の行の `#` を外して図を描きます。縦軸の `0.8` は 80% です。図だけでは分母の定義や件数は伝わらないため，上の表と合わせて読みます。

In [ ]:
# report["rate"].plot.bar(ylim=(0, 1))
# plt.show()

### 3. 数字を説明する（6 分）

一つの空港を選び，分子と分母を使って結果を一文にしてください。別の人がこの文だけを読んでも，期間・行先・分母が分かるように書きます。

- 見出し：
- 根拠：定時到着 ____ 便／分母 ____ 便，約 ____ %。
- その分母を選んだ理由：
- もう一方の依頼なら，分母に含める便はどう変わるか：

例えば「〇〇空港がよい」だけでは，何を比べたかが残りません。「2024 年 1 月の ____ 行きでは，____ 便のうち約 ____ % が定時に到着した」のように，数えた対象を入れます。

この結果を別の月の旅行に使う場合，さらに何を調べたいですか。今月の割合を計算したことと，今後の便の成績を予測したことは区別します。

追加で調べたいこと：

**提出前に**，ノートを最初から実行して保存します。選んだ行先・依頼，記入した分母の式，実行結果と上の説明を含めたノートを提出してください。任意の追加課題は，コメントのままで構いません。

### 追加課題（任意）

もう一方の依頼の率も計算し，同じ空港について比べます。`report` を上書きせず，別の名前へ保存してください。変わったのは定時到着便数か，分母か，両方かを説明します。順位が変わらなくても，差や件数から説明できます。

本文の重みの比較へ進みたい人は，MCO と SFO の率を半分ずつ重く見た場合と，実際の予定便数でまとめた場合を比べ，それぞれが何を答えるかを言葉にしてください。こちらは共通課題の提出には不要です。

In [ ]:
# 追加で調べるコードをここに書く（任意）